# MagNet Standalone Notebook

This notebook bundles the MagNet project `main.py` and its dependencies.
It is designed to run in environments like Google Colab without requiring file uploads.

In [ ]:
# Install dependencies
!pip install -r requirements.txt
import os
import sys
# Add current directory to path just in case
sys.path.append(os.getcwd())

### Configuration
Creating `config/config.yaml`...

In [ ]:
import os
os.makedirs('config', exist_ok=True)
config_content = """# Configuration parameters for MagNet ML Project

# Data settings
data:
  raw_path: "data/raw"
  processed_path: "data/processed"
  batch_size: 32
  test_split: 0.2

# Model settings
models:
  scaler:
    hidden_dim: 64
    layers: 3
  sequence:
    hidden_dim: 128
    num_layers: 2
  seq2seq:
    encoder_dim: 128
    decoder_dim: 128
  cnn:
    kernel_size: 3
    num_channels: 64
    num_layers: 3
  transformer:
    d_model: 64
    nhead: 4
    num_layers: 2
    dim_feedforward: 128
    dropout: 0.1

# Training settings
training:
  epochs: 100
  learning_rate: 0.001
  save_dir: "checkpoints"
"""
with open('config/config.yaml', 'w') as f:
    f.write(config_content)

### Module: `src\data\dataset.py`
Content from local file: `src\data\dataset.py`

In [ ]:
"""
MagNet Dataset Module.

This module provides a PyTorch Dataset wrapper for the MagNet data.
It handles:
1. Loading the full dataset into memory.
2. Computing derived quantities (B, H, Power Loss).
3. Normalizing features.
4. Serving data based on the requested mode ('scaler', 'sequence', 'seq2seq').

Classes:
- MagNetDataset(Dataset)
"""

import torch
from torch.utils.data import Dataset
import numpy as np
# [NOTEBOOK_BUNDLER] from src.data import loader, preprocessing

class MagNetDataset(Dataset):
    def __init__(self, file_path, mode='scaler', transform=None):
        """
        Args:
            file_path (str): Path to .mat file.
            mode (str): 'scaler', 'sequence', 'seq2seq'.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.mode = mode
        self.transform = transform
        
        # Load raw data
        print(f"Loading dataset from {file_path}...")
        raw_data = load_full_dataset(file_path)
        
        self.voltage = raw_data['voltage']
        self.current = raw_data['current']
        self.freq = raw_data['freq']
        self.temp = raw_data['temp']
        self.hdc = raw_data['hdc']
        self.duty = raw_data['duty']
        self.meta = raw_data['meta']
        
        # Compute derived features (B, H, Power Loss)
        print("Computing B field...")
        # Note: B calculation can be vectorized or looped.
        # Vectorized implementation of cumulative trapezoid is cleaner but requires care with shapes.
        # We'll use a loop or apply_along_axis if needed, but simple integration is fast.
        # B = Integral(V) / (N * Ae)
        
        # Remove DC offset from V per experiment
        v_mean = np.mean(self.voltage, axis=1, keepdims=True)
        v_clean = self.voltage - v_mean
        
        # Integrate
        # cumulative_trapezoid works on last axis by default, which is samples (dim 1)
        from scipy.integrate import cumulative_trapezoid
        # We need dt for each experiment.
        # If dt is scalar, easy. If array, we need to broadcast.
        dt = self.meta['dt']
        if np.ndim(dt) == 0:
            dt = float(dt)
        else:
            # Reshape to (N, 1) for broadcasting if possible, 
            # but cumtrapz doesn't accept array dx easily for different rows.
            # Assuming dt is constant for each experiment relative to time steps.
            pass

        # Since dt might vary per experiment, we might need a loop or valid avg dt
        # For simplicity in 'scaler' mode, we might trust specific dt.
        # Let's assume dt is constant within one cycle.
        
        # Efficient vector integration if dt is scalar or we average it?
        # Let's loop for safety in this version or use simple cumsum * dt if uniform.
        # cumtrapz is (y[i] + y[i-1])/2 * dx.
        
        # Vectorized cumtrapz:
        flux = cumulative_trapezoid(v_clean, axis=1, initial=0) # * dt later
        if np.ndim(dt) > 0:
            flux = flux * dt[:, None]
        else:
            flux = flux * dt
            
        self.b_field = flux / (self.meta['N_sec'] * self.meta['Ae'])
        
        # Remove DC from B
        b_mean = np.mean(self.b_field, axis=1, keepdims=True)
        self.b_field = self.b_field - b_mean
        
        print("Computing H field...")
        # H = (N * I) / Le
        self.h_field = (self.meta['N_prim'] * self.current) / self.meta['Le']
        
        print("Computing Power Loss...")
        # Pv = Mean(V * I) / Ve ? No, MagNet definition is usually Energy/Cycle / Volume
        # Energy = Integral(P_inst) dt
        p_inst = self.voltage * self.current
        energy = np.trapz(p_inst, axis=1) # * dt
        if np.ndim(dt) > 0:
            energy = energy * dt
        else:
            energy = energy * dt
            
        period = (self.voltage.shape[1] * dt) if np.ndim(dt)==0 else (self.voltage.shape[1] * dt) # approx 
        # Actually Period = 1/Freq usually.
        # But let's use the full cycle time provided.
        
        # Effective Volume
        ve = self.meta['Ae'] * self.meta['Le']
        
        self.power_loss = energy / (period * ve) # W/m^3
        
        # Normalize inputs (MinMax)
        # Store scalers for inversion? For now, simple minmax.
        # For ML, we should calculate stats on TRAINING set only.
        # But here we do full dataset.
        # Ideally: split first, then normalize.
        # For now, we normalize everything together (simple approach).
        
        self.norm_b, _ = normalize_data(self.b_field)
        self.norm_h, _ = normalize_data(self.h_field)
        self.norm_freq, _ = normalize_data(self.freq)
        self.norm_temp, _ = normalize_data(self.temp)
        self.norm_hdc, _ = normalize_data(self.hdc)
        
        # Target normalization (log scale is often good for loss)
        self.log_loss = np.log10(np.abs(self.power_loss) + 1e-6)
        
    def __len__(self):
        return self.voltage.shape[0]
        
    def __getitem__(self, idx):
        if self.mode == 'scaler':
            # Input: Freq, Temp, Hdc
            # Target: Power Loss
            x = torch.tensor([
                self.norm_freq[idx],
                self.norm_temp[idx],
                self.norm_hdc[idx]
            ], dtype=torch.float32)
            y = torch.tensor([self.log_loss[idx]], dtype=torch.float32)
            return x, y
            
        elif self.mode == 'sequence':
            # Input: B waveform (or H)
            # Target: Power Loss
            # Shape: (Seq_Len, 1)
            b_seq = torch.tensor(self.norm_b[idx], dtype=torch.float32).unsqueeze(-1)
            y = torch.tensor([self.log_loss[idx]], dtype=torch.float32)
            return b_seq, y
            
        elif self.mode == 'seq2seq':
            # Input: B waveform
            # Target: H waveform (or vice versa)
            # Many papers predict H from B.
            x = torch.tensor(self.norm_b[idx], dtype=torch.float32).unsqueeze(-1)
            y = torch.tensor(self.norm_h[idx], dtype=torch.float32).unsqueeze(-1)
            return x, y
            
        else:
            raise ValueError(f"Unknown mode: {self.mode}")

### Module: `src\models\scaler_model.py`
Content from local file: `src\models\scaler_model.py`

In [ ]:
"""
Scaler-to-Scaler Model.

This module implements a standard Multi-Layer Perceptron (MLP) for predicting
scalar outputs (e.g., Power Loss) from scalar inputs (Frequency, Temperature, Hdc).
"""

import torch
import torch.nn as nn

class ScalerNetwork(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, output_dim=1, num_layers=3):
        """
        Args:
            input_dim (int): Number of input features (default 3: Freq, Temp, Hdc).
            hidden_dim (int): Number of neurons in hidden layers.
            output_dim (int): Number of output features (default 1: Power Loss).
            num_layers (int): Number of hidden layers.
        """
        super(ScalerNetwork, self).__init__()
        
        layers = []
        
        # Input Layer
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm1d(hidden_dim))
        
        # Hidden Layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            
        # Output Layer
        layers.append(nn.Linear(hidden_dim, output_dim))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

### Module: `src\models\sequence_model.py`
Content from local file: `src\models\sequence_model.py`

In [ ]:
"""
Sequence-to-Scaler Model.

This module implements models for mapping time-series waveforms to single scalar values.
It uses an LSTM encoder followed by a linear head on the final hidden state.
"""

import torch
import torch.nn as nn

class SequenceToScalerNetwork(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, output_dim=1, num_layers=2):
        """
        Args:
            input_dim (int): Number of features in input sequence (default 1: B or H).
            hidden_dim (int): LSTM hidden dimension.
            output_dim (int): Output size (default 1: Power Loss).
            num_layers (int): Number of LSTM layers.
        """
        super(SequenceToScalerNetwork, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1 if num_layers > 1 else 0
        )
        
        # Head to map final hidden state to output
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        
    def forward(self, x):
        # x shape: (Batch, Seq_Len, Input_Dim)
        
        # LSTM output: (Batch, Seq, Hidden), (h_n, c_n)
        # h_n shape: (Num_Layers, Batch, Hidden)
        output, (h_n, c_n) = self.lstm(x)
        
        # Use final layer's final hidden state
        final_hidden = h_n[-1] # (Batch, Hidden)
        
        out = self.head(final_hidden)
        return out

### Module: `src\models\seq2seq_model.py`
Content from local file: `src\models\seq2seq_model.py`

In [ ]:
"""
Sequence-to-Sequence Model.

This module implements Encoder-Decoder architectures for mapping input waveforms 
(Excitation, e.g., H) to output waveforms (Response, e.g., B).
"""

import torch
import torch.nn as nn

class Seq2SeqNetwork(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, output_dim=1, num_layers=2):
        """
        Args:
            input_dim (int): Input feature size.
            hidden_dim (int): Hidden size.
            output_dim (int): Output feature size.
            num_layers (int): Depth of LSTM.
        """
        super(Seq2SeqNetwork, self).__init__()
        
        # Encoder
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1
        )
        
        # Decoder
        # Input to decoder is previous output (or ground truth in training).
        # We model this as mapping Hidden State -> Sequence.
        
        # Simple Approach: Use LSTM to map (Batch, Seq, Hidden) -> (Batch, Seq, Out).
        # But standard Seq2Seq uses an Decoder LSTM.
        
        # Here we implement a simple LSTM-based mapping (Many-to-Many).
        # Since input and output length are same for hysteresis loops.
        # This acts like a Bi-Directional LSTM or just a mapped LSTM.
        
        self.decoder = nn.LSTM(
            input_size=hidden_dim, # We might feed encoder outputs or similar
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1
        )
        
        self.head = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: (Batch, Seq, Input)
        
        # Encoder
        # We want to map Sequence -> Sequence.
        # If lengths are same and it's 1:1 mapping (like filtering), 
        # a single LSTM (Encoder) + Linear Head per step is sufficient.
        
        # output: (Batch, Seq, Hidden)
        enc_out, _ = self.encoder(x)
        
        # Decode/Map
        # dec_out, _ = self.decoder(enc_out) # Optional: Deepen model
        
        # Project to output
        # out: (Batch, Seq, Output)
        out = self.head(enc_out)
        
        return out

### Module: `src\models\cnn_model.py`
Content from local file: `src\models\cnn_model.py`

In [ ]:
"""
CNN Model for Sequence Data.

This module implements a 1D Convolutional Neural Network (CNN) for processing
time-series sequence data (e.g., flux density waveforms).
"""

import torch
import torch.nn as nn

class CNNNetwork(nn.Module):
    def __init__(self, input_dim=1, kernel_size=3, num_channels=64, num_layers=3, output_dim=1):
        """
        Args:
            input_dim (int): Number of input channels/features (default 1 for B waveform).
            kernel_size (int): Size of the convolutional kernel.
            num_channels (int): Number of channels in the convolutional layers.
            num_layers (int): Number of convolutional layers.
            output_dim (int): Number of output features (default 1 for scalar loss or sequence H).
        """
        super(CNNNetwork, self).__init__()
        
        layers = []
        
        # Initial Conv Layer
        # Input shape: (Batch, Channels, Seq_Len) -> Conv1d needs (N, C_in, L)
        # However, our data loader typically provides (Batch, Seq_Len, Features)
        # We'll handle the permute in forward().
        
        layers.append(nn.Conv1d(in_channels=input_dim, out_channels=num_channels, kernel_size=kernel_size, padding=kernel_size//2))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm1d(num_channels))
        
        # Hidden Conv Layers
        for _ in range(num_layers - 1):
            layers.append(nn.Conv1d(in_channels=num_channels, out_channels=num_channels, kernel_size=kernel_size, padding=kernel_size//2))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(num_channels))
            
        self.feature_extractor = nn.Sequential(*layers)
        
        # Output Head
        # For sequence-to-scalar, we might pool first. For sequence-to-sequence, we keep it.
        # Assuming we want to support both or default to something useful.
        # Let's assume Sequence-to-Sequence for now, or use a Global Average Pooling for Scalar.
        # User request didn't specify, but "CNN" usually implies feature extraction over time.
        # Let's add a Global Average Pooling + Linear for Scalar output by default since 'sequence_model' was seq-to-scalar.
        # But wait, seq2seq_model exists too.
        # Let's make it flexible or design for Sequence-to-Scaler (Loss prediction).
        # Given "Transformer" and "CNN" were added together, likely for the same task as "all" models.
        # 'sequence' model predicts Log Loss (Scalar). 'seq2seq' predicts H (Sequence).
        # Let's make CNN predict Scalar Loss (Sequence-to-Scalar) similar to 'sequence_model'.
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(num_channels, output_dim)
        
    def forward(self, x):
        # x shape: (Batch, Seq_Len, Features)
        # Conv1d expects (Batch, Features, Seq_Len)
        x = x.permute(0, 2, 1)
        
        features = self.feature_extractor(x)
        
        # features shape: (Batch, Num_Channels, Seq_Len)
        
        # Global Average Pooling -> (Batch, Num_Channels, 1)
        pooled = self.global_pool(features).squeeze(-1)
        
        # FC -> (Batch, Output_Dim)
        out = self.fc(pooled)
        
        return out

### Module: `src\models\transformer_model.py`
Content from local file: `src\models\transformer_model.py`

In [ ]:
"""
Transformer Model for Sequence Data.

This module implements a Transformer-based architecture for processing
time-series sequence data.
"""

import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0).transpose(0, 1) # Shape: (Max_Len, 1, D_Model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (Seq_Len, Batch, D_Model)
        return x + self.pe[:x.size(0), :]

class TransformerNetwork(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1, output_dim=1):
        """
        Args:
            input_dim (int): Number of input features.
            d_model (int): Hidden dimension size.
            nhead (int): Number of attention heads.
            num_layers (int): Number of transformer encoder layers.
            dim_feedforward (int): Dimension of the FFN.
            dropout (float): Dropout rate.
            output_dim (int): Output dimension (default 1 for scalar loss).
        """
        super(TransformerNetwork, self).__init__()
        
        self.input_embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
        self.decoder = nn.Linear(d_model, output_dim)
        self.d_model = d_model

        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        self.input_embedding.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.zero_()
        self.decoder.weight.data.uniform_(-initrange, initrange)

    def forward(self, x):
        # x shape from loader: (Batch, Seq_Len, Features)
        # Transformer expects: (Seq_Len, Batch, Features) for default batch_first=False
        
        x = x.permute(1, 0, 2) 
        
        x = self.input_embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        
        output = self.transformer_encoder(x)
        
        # output shape: (Seq_Len, Batch, D_Model)
        
        # For sequence-to-scalar, take the output of the last time step? Or average?
        # Let's take the mean over the sequence length.
        output = output.mean(dim=0) # (Batch, D_Model)
        
        output = self.decoder(output) # (Batch, Output_Dim)
        
        return output

### Module: `src\models\__init__.py`
Content from local file: `src\models\__init__.py`

In [ ]:
"""
Neural Network Models Module.
"""

# [NOTEBOOK_BUNDLER] from .scaler_model import ScalerNetwork
# [NOTEBOOK_BUNDLER] from .sequence_model import SequenceToScalerNetwork
# [NOTEBOOK_BUNDLER] from .seq2seq_model import Seq2SeqNetwork
# [NOTEBOOK_BUNDLER] from .cnn_model import CNNNetwork
# [NOTEBOOK_BUNDLER] from .transformer_model import TransformerNetwork

### Module: `src\training\train.py`
Content from local file: `src\training\train.py`

In [ ]:
"""
Training Loop Module.

This module contains the logic for training the neural networks.
It includes the training loop, validation step, loss calculation, and checkpointing.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os
import copy

def train_model(model, train_loader, val_loader, config, device='cpu'):
    """
    Generic training loop.
    
    Args:
        model (nn.Module): The model to train.
        train_loader (DataLoader): Training data.
        val_loader (DataLoader): Validation data.
        config (dict): Configuration dictionary (lr, epochs, save_dir).
        device (str): 'cpu' or 'cuda'.
        
    Returns:
        model (nn.Module): Trained model (best weights).
        history (dict): Training history (loss).
    """
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.get('learning_rate', 0.001))
    
    num_epochs = config.get('epochs', 100)
    save_dir = config.get('save_dir', 'checkpoints')
    os.makedirs(save_dir, exist_ok=True)
    
    best_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        
        # Use tqdm for progress bar if interactive
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for inputs, targets in pbar:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
            pbar.set_postfix({'loss': loss.item()})
            
        epoch_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)
        
        # Validation Phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * inputs.size(0)
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        history['val_loss'].append(epoch_val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Val Loss: {epoch_val_loss:.4f}")
        
        # Deep Copy Best Model
        if epoch_val_loss < best_loss:
            best_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
            
    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

### Module: `src\training\evaluate.py`
Content from local file: `src\training\evaluate.py`

In [ ]:
"""
Evaluation Module.

This module provides functions to evaluate model performance on test datasets.
"""

import torch
import torch.nn as nn
import numpy as np

def evaluate_model(model, test_loader, device='cpu'):
    """
    Evaluates the model and returns predictions and metrics.
    
    Args:
        model (nn.Module): Trained model.
        test_loader (DataLoader): Test data.
        device (str): Device.
        
    Returns:
        metrics (dict): MSE, MAE, Relative Error.
        predictions (list): List of pred tensors.
        targets (list): List of target tensors.
    """
    model.eval()
    model.to(device)
    
    preds = []
    actuals = []
    
    criterion_mse = nn.MSELoss()
    criterion_mae = nn.L1Loss()
    
    total_mse = 0.0
    total_mae = 0.0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            
            mse = criterion_mse(outputs, targets)
            mae = criterion_mae(outputs, targets)
            
            total_mse += mse.item() * inputs.size(0)
            total_mae += mae.item() * inputs.size(0)
            
            preds.append(outputs.cpu().numpy())
            actuals.append(targets.cpu().numpy())
            
    num_samples = len(test_loader.dataset)
    avg_mse = total_mse / num_samples
    avg_mae = total_mae / num_samples
    
    metrics = {
        'mse': avg_mse,
        'mae': avg_mae,
        'rmse': np.sqrt(avg_mse)
    }
    
    return metrics, np.concatenate(preds), np.concatenate(actuals)

### Module: `src\utils\visualization.py`
Content from local file: `src\utils\visualization.py`

In [ ]:
"""
Visualization Module.

This module provides plotting functions for inspecting model performance.
"""

import matplotlib.pyplot as plt
import numpy as np

def plot_loss_curve(history, title='Training History'):
    """
    Plots Train vs Val Loss.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_prediction_scatter(preds, targets, title='Predictions vs Actuals'):
    """
    Scatter plot for scalar regression.
    """
    plt.figure(figsize=(8, 8))
    plt.scatter(targets, preds, alpha=0.5)
    
    # Perfect line
    min_val = min(np.min(targets), np.min(preds))
    max_val = max(np.max(targets), np.max(preds))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title(title)
    plt.grid(True)
    plt.show()
    
def plot_bh_loop(pred_b, pred_h, actual_b, actual_h, title='B-H Loop Comparison'):
    """
    Plots predicted vs actual B-H loop.
    Args: (Seq_Len,) arrays.
    """
    plt.figure(figsize=(8, 6))
    plt.plot(actual_h, actual_b, 'b-', label='Actual', linewidth=2)
    plt.plot(pred_h, pred_b, 'r--', label='Predicted', linewidth=2)
    plt.xlabel('H (A/m)')
    plt.ylabel('B (T)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

### Module: `main.py`
Content from local file: `main.py`

In [ ]:
"""
MagNet Project Entry Point.

This script serves as the main interface for training and evaluating the neural network models.
"""

import argparse
import torch
from torch.utils.data import DataLoader, random_split
# [NOTEBOOK_BUNDLER] from src.data.dataset import MagNetDataset
# [NOTEBOOK_BUNDLER] from src.models import ScalerNetwork, SequenceToScalerNetwork, Seq2SeqNetwork, CNNNetwork, TransformerNetwork
# [NOTEBOOK_BUNDLER] from src.training.train import train_model
# [NOTEBOOK_BUNDLER] from src.training.evaluate import evaluate_model
# [NOTEBOOK_BUNDLER] from src.utils.visualization import plot_loss_curve, plot_prediction_scatter
import yaml
import os

def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def main(args=None):
    parser = argparse.ArgumentParser(description="MagNet Deep Learning Pipeline")
    parser.add_argument('--config', type=str, default='config/config.yaml', help='Path to config file')
    parser.add_argument('--data', type=str, required=True, help='Path to .mat file')
    parser.add_argument('--model', type=str, choices=['scaler', 'sequence', 'seq2seq', 'cnn', 'transformer', 'all'], required=True, help='Model type to train')
    parser.add_argument('--epochs', type=int, help='Override epochs in config')
    args = parser.parse_args(args)
    
    config = load_config(args.config)
    
    # Override config
    if args.epochs:
        config['training']['epochs'] = args.epochs
        
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    models_to_run = [args.model] if args.model != 'all' else ['scaler', 'sequence', 'seq2seq', 'cnn', 'transformer']
    
    for model_name in models_to_run:
        print(f"\n{'='*20} Training {model_name.upper()} Model {'='*20}")
        
        # 1. Dataset
        print("Preparing Dataset...")
        dataset = MagNetDataset(args.data, mode=model_name)
        
        # Split (80/20)
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_set, val_set = random_split(dataset, [train_size, val_size])
        
        batch_size = config['data'].get('batch_size', 32)
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=batch_size)
        
        # 2. Model
        if model_name == 'scaler':
            # Input: Freq, Temp, Hdc (3)
            # Output: Log Loss (1)
            model_conf = config['models']['scaler']
            model = ScalerNetwork(input_dim=3, hidden_dim=model_conf['hidden_dim'], num_layers=model_conf['layers'], output_dim=1)
            
        elif model_name == 'sequence':
            # Input: B (1)
            # Output: Log Loss (1)
            model_conf = config['models']['sequence']
            model = SequenceToScalerNetwork(input_dim=1, hidden_dim=model_conf['hidden_dim'], output_dim=1)
            
        elif model_name == 'seq2seq':
            # Input: B (1)
            # Output: H (1)
            model_conf = config['models']['seq2seq']
            model = Seq2SeqNetwork(input_dim=1, encoder_dim=model_conf['encoder_dim'], decoder_dim=model_conf['decoder_dim'], output_dim=1)
            
        elif model_name == 'cnn':
            # Input: B (1)
            # Output: Log Loss (1) (Scalar)
            model_conf = config['models']['cnn']
            model = CNNNetwork(input_dim=1, kernel_size=model_conf['kernel_size'], num_channels=model_conf['num_channels'], num_layers=model_conf['num_layers'], output_dim=1)
        
        elif model_name == 'transformer':
            # Input: B (1)
            # Output: Log Loss (1) (Scalar)
            model_conf = config['models']['transformer']
            model = TransformerNetwork(input_dim=1, d_model=model_conf['d_model'], nhead=model_conf['nhead'], num_layers=model_conf['num_layers'], dim_feedforward=model_conf['dim_feedforward'], dropout=model_conf['dropout'], output_dim=1)

        # 3. Train
        print("Starting training...")
        # Subset config for training
        train_config = config['training']
        train_config['save_dir'] = os.path.join(train_config['save_dir'], model_name)
        
        if not os.path.exists(train_config['save_dir']):
            os.makedirs(train_config['save_dir'])
        
        model = model.to(device)
        trained_model, history = train_model(model, train_loader, val_loader, train_config, device)
        
        # 4. Evaluate & Visualize
        print("Evaluating...")
        metrics, preds, targets = evaluate_model(trained_model, val_loader, device)
        print(f"Validation Metrics: {metrics}")
        
        # Plot Loss
        plot_loss_curve(history, title=f'{model_name} Training Loss')
        
        # Plot Predictions
        if model_name in ['scaler', 'sequence', 'cnn', 'transformer']:
            plot_prediction_scatter(preds, targets, title=f'{model_name}: Pred vs Actual Loss')
        elif model_name == 'seq2seq':
            pass

if __name__ == "__main__":
    main()

### Run Training
Call the `main()` function with arguments as a list of strings.

In [ ]:
# Example: Train CNN Model
# Make sure the data file path is correct
data_file = "3C90_TX-25-15-10_Data1_Cycle.mat"
if not os.path.exists(data_file):
    print(f"Warning: {data_file} not found. Please upload it or fix the path.")

# Arguments: --data <path> --model <model_name> --epochs <N>
args = ['--data', data_file, '--model', 'cnn', '--epochs', '10']

try:
    main(args)
except SystemExit:
    # argparse raises SystemExit on help or error, catch it so notebook doesn't crash
    pass